In [33]:
import torch
import torch.optim as optim
from torch import nn
import numpy as np

device = torch.device('cpu')

In [2]:
# Standard Tensor
x_data = torch.tensor([[1.,2.], [3., 4.]])

# A parameter tensor (requires_autorad = True, Need gradients)
w = torch.tensor([[1.,2.], [3., 4.]], requires_grad=True)

In [3]:
a = torch.tensor(2.0, requires_grad=True)
b = torch.tensor(3.0, requires_grad=True)
x = torch.tensor(4.0, requires_grad=True)

y = a+b
z= x*y

z

tensor(20., grad_fn=<MulBackward0>)

In [4]:
# for matrix multiplication tye rule is that m1 columns = m2 rows
# No. of columns on the first matrix must be equal to 
# the no. of rows on the second matrix
# @ is the symbol for matrix multiplicaion

# Shape: (2,3)
m1 = torch.tensor([[1,2,3], [4,5,6]])

# Shape: (3,2)
m2 = torch.tensor([[7,8],[9,10], [11,12]])

matrix_calcu = m1@m2

matrix_calcu

tensor([[ 58,  64],
        [139, 154]])

In [5]:
# Reduction Operators
# calculate averages by dimension

scores = torch.tensor([[10.,20.,30.], [5.,10.,15.]])

#  To get the average for each assignment, we collapse the student dimension(dim=10)
avg_per_assignment = scores.mean(dim=0)

#  To get the average for each Student, we collapse the assignment dimension(dim=10)
avg_per_student = scores.mean(dim=1)

avg_per_assignment, avg_per_student

# dim=0 → operate down the rows (along the first axis)
# dim=1 → operate across the columns (along the second axis

(tensor([ 7.5000, 15.0000, 22.5000]), tensor([20., 10.]))

In [6]:
xy = torch.arange(12).reshape(3,4)

col_2 = xy[:, 3]
row_3 = xy[2,:]
col_2, row_3

(tensor([ 3,  7, 11]), tensor([ 8,  9, 10, 11]))

In [7]:
#  Argmax

scores2 = torch.tensor([[10,0,5,20,1], [1,30,2,5,0]])
best_indices = torch.argmax(scores2, dim=1)
best_indices

tensor([3, 1])

In [8]:
# torch.gather

data = torch.tensor([
    [10,11,12,13],
    [20,21,22,23],
    [30,31,32,33]
])

indices_to_select = torch.tensor([[2], [0], [3]])

selected_values = torch.gather(data, dim=1, index=indices_to_select)
selected_values

tensor([[12],
        [20],
        [33]])

In [9]:
#  Forward Pass
# ŷ =XW+b
# X is our input data, W is the weight, b is the bias and ŷ is the prediction or guess
# W and b are the 2 knobs our model can turn

# Our batch of data will have 10 data points
N=10
# Each data poin has 1 input feature and 1 output value
D_in = 1
D_out = 1

# Create our input data X
X = torch.randn(N, D_in)

# Create our true target labels y by using the 'true' W and b
#  The 'true' W is 2.0, the 'true' b is 1.0

true_W = torch.tensor([[2.0]])
true_b = torch.tensor([[1.0]])

y_true = X @ true_W + true_b + torch.randn(N, D_out) * 0.1 # Add a little noise

# Initialize our parameters with random values
# shapes must be correct for matrix multiplication

W = torch.randn(D_in, D_out, requires_grad=True)
b = torch.randn(1, requires_grad=True)

W, b

y_hat = X @ W + b

y_hat, y_true

(tensor([[0.2437],
         [0.2995],
         [0.4294],
         [0.2470],
         [0.4085],
         [0.3399],
         [0.2056],
         [0.2574],
         [0.2934],
         [0.2571]], grad_fn=<AddBackward0>),
 tensor([[ 2.3779],
         [ 1.0645],
         [-2.9677],
         [ 2.5232],
         [-2.4615],
         [-0.4107],
         [ 3.6221],
         [ 2.1710],
         [ 0.9819],
         [ 2.0666]]))

In [10]:
# Backward Pass

# lets calculate Loss
error = y_hat - y_true
squared_error = error ** 2
loss = squared_error.mean()

loss

tensor(4.9744, grad_fn=<MeanBackward0>)

In [11]:
loss.backward()

In [12]:
W.grad, b.grad

(tensor([[-4.4463]]), tensor([-1.1972]))

In [32]:
# The entire learning process

learning_rate, epochs = 0.01, 100

W,b = torch.randn(1,1, requires_grad=True), torch.randn(1, requires_grad=True)

# Training loop
for epoch in range(epochs):
    # forward pass and loss
    y_hat = X @ W + b
    loss = torch.mean((y_hat - y_true)**2)
    # Backward Pass
    loss.backward()

    # Update Parameters
    with torch.no_grad():
        W -= learning_rate * W.grad; b -= learning_rate * b.grad

    # Zero gradients
    W.grad.zero_(); b.grad.zero_()

    if epoch % 10 == 0:
        print(f"Epoch {epoch:02d}: Loss={loss.item():.4f}, W={W.item():.3f}, b={b.item():.3f}")

print(f"\nFinal Parameters: W={W.item():.3f}, b={b.item():.3f}")
print("True Parameters: W=2.000, b=1.000")

Epoch 00: Loss=5.9962, W=-0.437, b=-0.594
Epoch 10: Loss=4.4468, W=-0.245, b=-0.262
Epoch 20: Loss=3.5602, W=-0.121, b=0.001
Epoch 30: Loss=3.0374, W=-0.044, b=0.211
Epoch 40: Loss=2.7199, W=0.001, b=0.380
Epoch 50: Loss=2.5216, W=0.027, b=0.515
Epoch 60: Loss=2.3947, W=0.039, b=0.625
Epoch 70: Loss=2.3118, W=0.043, b=0.715
Epoch 80: Loss=2.2567, W=0.042, b=0.788
Epoch 90: Loss=2.2197, W=0.038, b=0.847

Final Parameters: W=0.033, b=0.892
True Parameters: W=2.000, b=1.000


In [27]:
# Using torch.nn
import torch.nn as nn

# Input feature and Output feature
D_in = 1
D_out = 1

# create linear layer
linear_layer = nn.Linear(in_features=D_in, out_features=D_out)
# linear_layer.weight and linear_layer.bias to check the weight and bias created by nn

# prediction
X = torch.randn(10,1)
y_hat = linear_layer(X)
y_hat

tensor([[ 0.5719],
        [-0.9411],
        [-2.6835],
        [-0.4093],
        [ 1.4117],
        [ 0.4237],
        [-0.9195],
        [-1.9904],
        [ 0.4150],
        [-0.8874]], grad_fn=<AddmmBackward0>)

In [15]:
# NN.ReLU
relu = nn.ReLU()
relu(y_hat)

tensor([[0.0000],
        [0.0000],
        [0.0000],
        [0.0000],
        [0.0000],
        [0.0000],
        [0.0000],
        [0.0000],
        [0.0000],
        [0.0626]], grad_fn=<ReluBackward0>)

In [16]:
# NN.GELU
gelu = nn.GELU()
gelu(y_hat)

tensor([[-0.1591],
        [-0.1014],
        [-0.1589],
        [-0.1483],
        [-0.1488],
        [-0.1335],
        [-0.1657],
        [-0.1502],
        [-0.1696],
        [ 0.0328]], grad_fn=<GeluBackward0>)

In [25]:
# NN.SOFTMAX
# almost used on the final output layer for classification
#  converts logits to probability distribution

softmax = nn.Softmax(dim=-1)
# Raw ,odel scores for 2 items, accroiss 4 possible classes
logits = torch.tensor([[1.0,3.0,0.5,1.5], [-1.0,2.0,1.0,0.0]])
probabilities = softmax(logits)

probabilities, probabilities[1].sum(), probabilities[0].sum()

(tensor([[0.0939, 0.6942, 0.0570, 0.1549],
         [0.0321, 0.6439, 0.2369, 0.0871]]),
 tensor(1.0000),
 tensor(1.))

In [18]:
# NN.EMBEDDING

vocab_size = 10 # language has 10 unique words
embedding_dim = 3 # we will represent each word with 3d vector

embedding_layer = nn.Embedding(vocab_size, embedding_dim)

# input: a sentence where each word is an id. e.g, 1,5,0,8
input_ids = torch.tensor([[1,5,0,8]])
word_vectors = embedding_layer(input_ids)
word_vectors

tensor([[[-1.2298,  0.9052, -0.4449],
         [-1.7006,  0.4500,  0.5824],
         [-0.6493, -1.2583, -1.4197],
         [ 0.1983,  0.8846, -0.6419]]], grad_fn=<EmbeddingBackward0>)

In [19]:
# NN.LAYERNORM

# our word vectors have a feature dimension of 3
norm_layer = nn.LayerNorm(normalized_shape=3)
input_features = torch.tensor([[[1.0,2.0,3.0], [4.0,5.0,6.0]]])
normalized_features = norm_layer(input_features)

normalized_features.mean(dim=-1), normalized_features.std(dim=-1)

(tensor([[0., 0.]], grad_fn=<MeanBackward1>),
 tensor([[1.2247, 1.2247]], grad_fn=<StdBackward0>))

In [20]:
# NN.DROPOUT

dropout_layer = nn.Dropout(p=0.5)
input_tensor = torch.ones(1,10)

# Activate dropout training
dropout_layer.train()
output_during_train = dropout_layer(input_tensor)

# Deactivate dropout for evaluation/training
dropout_layer.eval()
output_during_eval = dropout_layer(input_tensor)

output_during_train, output_during_eval

(tensor([[2., 0., 0., 2., 2., 2., 0., 0., 2., 0.]]),
 tensor([[1., 1., 1., 1., 1., 1., 1., 1., 1., 1.]]))

In [21]:
# More Elegant way

class LinearRegressionModel(nn.Module):
    def __init__(self, in_features, out_features):
        super().__init__()
        self.linear_layer = nn.Linear(in_features, out_features)

    def forward(self, x):
        return self.linear_layer(x)

model = LinearRegressionModel(1,1)
model

LinearRegressionModel(
  (linear_layer): Linear(in_features=1, out_features=1, bias=True)
)

In [22]:
# The optimizer torch.optim

# Hyperparameters
learning_rate = 0.01

# Created an Adam Optimizer
# we pass model.parameters() to tell it which sensors to manage

optimizer = optim.Adam(model.parameters(), lr=learning_rate)

# we will also grab a pre-built loss function from torch.nn
loss_fn = nn.MSELoss()

In [23]:
# The entire learning process

learning_rate, epochs = 0.01, 100
optimizer = optim.Adam(model.parameters(), lr=learning_rate)
# model = nn.Linear(in_features=D_in, out_features=D_out)
loss_fn = nn.MSELoss()
y_true = X @ true_W + true_b + torch.randn(N, D_out) * 0.1 # Add a little noise

# Each data poin has 1 input feature and 1 output value
D_in = 1
D_out = 1

# Create our input data X
X = torch.randn(10,1)


# Training loop
for epoch in range(epochs):
    # forward pass and loss
    y_hat = model(X)

    # Calculate loss
    loss = loss_fn(y_hat, y_true)

    # Three line mantra
    # 1. Zero the gradients
    optimizer.zero_grad()
    # 2. Compute gradients
    loss.backward()
    # 3. Update the parameters
    optimizer.step()
    
    # Optional: print progress
    if epoch % 5 == 0:
        print(f"Epoch {epoch:02d}: Loss={loss.item():.4f}")

Epoch 00: Loss=2.3571
Epoch 05: Loss=2.2943
Epoch 10: Loss=2.2458
Epoch 15: Loss=2.2105
Epoch 20: Loss=2.1851
Epoch 25: Loss=2.1652
Epoch 30: Loss=2.1483
Epoch 35: Loss=2.1340
Epoch 40: Loss=2.1225
Epoch 45: Loss=2.1141
Epoch 50: Loss=2.1082
Epoch 55: Loss=2.1043
Epoch 60: Loss=2.1016
Epoch 65: Loss=2.0999
Epoch 70: Loss=2.0988
Epoch 75: Loss=2.0983
Epoch 80: Loss=2.0980
Epoch 85: Loss=2.0978
Epoch 90: Loss=2.0978
Epoch 95: Loss=2.0978
